# Layer-bias MCTS experiment

This notebook runs the layer-bias diagnosis and three-way correction comparison for TP-MCTS on Google Colab. It applies all instrumentation as in-memory runtime patches, so the cloned repository is not edited.

**Colab setup** matches `experiments.ipynb`: the first code cell clones `https://github.com/eliezerRevach/tp_mcts.git` into `/content/tp_mcts` when `/content` exists (no manual URL). **Local use:** run from your checkout; the same cell is a no-op and resolves the repo root from the current working directory.

In [ ]:
# Cell 1 - Setup (Colab: same pattern as experiments.ipynb)
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Same default clone as root experiments.ipynb — edit REPO_URL only if you use a fork.
REPO_DIR = "/content/tp_mcts"
REPO_URL = "https://github.com/eliezerRevach/tp_mcts.git"

IN_COLAB = os.path.isdir("/content")

# Colab: fresh clone every time (avoids stale /content/tp_mcts). No-op if /content missing.
if not IN_COLAB:
    print("Skip Colab clone: not on Colab (`/content` missing). cwd=", os.getcwd())
else:
    if not REPO_DIR.startswith("/content/"):
        raise RuntimeError(f"Refusing to delete unexpected path: {REPO_DIR}")
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.chdir("/content")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "-q",
            "install",
            "dill",
            "numpy",
            "pandas",
            "openpyxl",
            "matplotlib",
            "tqdm",
        ]
    )
    print("Ready (Colab):", os.getcwd())


def _has_unified(path: Path) -> bool:
    return (path / "unified_planning").is_dir()


colab_root = Path(REPO_DIR)
if IN_COLAB and colab_root.is_dir() and _has_unified(colab_root):
    repo_root = colab_root.resolve()
else:
    start = Path.cwd().resolve()
    repo_root = None
    for p in [start, *start.parents]:
        if _has_unified(p):
            repo_root = p.resolve()
            break
    if repo_root is None:
        raise RuntimeError(
            "Could not find repo root (folder containing unified_planning/). "
            "On Colab, run the clone block above first. Locally, `cd` to the repository."
        )

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

ARTIFACTS_DIR = Path("/content/artifacts") if IN_COLAB else (repo_root / "artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Solver / benchmark constants. QUICK_MODE should finish quickly on a Colab CPU.
QUICK_MODE = True
N_PROBLEMS = 5 if QUICK_MODE else None
N_RUNS = 3 if QUICK_MODE else 10
SEED = 123
SEARCH_TIME = 0.25 if QUICK_MODE else 1.0
SEARCH_DEPTH = 40
K = 10
SELECTION_TYPE = "avg"
HEURISTIC_NAME = "temporal_probabilistic_rpg"
TEMPORAL_STRATEGY = "baseline"
TEMPORAL_HEURISTIC_DEPTH = 25
STEP_LIMIT = 90
EXPLORATION_CONSTANT = 10.0
DISCOUNT_FACTOR = 0.95
REWARD_MODE = "deadline"
STEP_PENALTY = -0.05
ONLINE_WARMUP_VISITS = 100

requirements = repo_root / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
else:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "dill",
            "numpy",
            "pandas",
            "matplotlib",
            "tqdm",
        ],
        check=True,
    )

print(f"Repo root: {repo_root}")
print(f"Artifacts: {ARTIFACTS_DIR}")
print(f"QUICK_MODE={QUICK_MODE}, N_RUNS={N_RUNS}, SEARCH_TIME={SEARCH_TIME}")

In [ ]:
# Cell 2 - Runtime patches for logging and layer-bias correction
import csv
import math
import random
import time
from pathlib import Path
from collections import defaultdict

import unified_planning as up
from unified_planning.shortcuts import *  # noqa: F401,F403 - initializes local unified_planning imports.
import unified_planning.engines.solvers.mcts as mcts_mod
from unified_planning.engines.utils import create_init_stn, update_stn

LOG_COLUMNS = [
    "event_type",
    "problem_id",
    "trial_id",
    "mode",
    "iteration",
    "depth",
    "h_value",
    "current_time",
    "eventual_return",
]

LAYER_BIAS_EPISODE_METRICS = []


class RunningDepthStats:
    """Welford running stats keyed by tree depth."""

    def __init__(self):
        self.stats = defaultdict(lambda: {"count": 0, "mean": 0.0, "M2": 0.0})

    def update(self, depth, value):
        if depth is None:
            return
        depth = int(depth)
        value = float(value)
        s = self.stats[depth]
        s["count"] += 1
        delta = value - s["mean"]
        s["mean"] += delta / s["count"]
        s["M2"] += delta * (value - s["mean"])

    def mean(self, depth):
        return self.stats[int(depth)]["mean"]

    def count(self, depth):
        return self.stats[int(depth)]["count"]

    def std(self, depth):
        s = self.stats[int(depth)]
        if s["count"] <= 1:
            return 0.0
        return math.sqrt(s["M2"] / (s["count"] - 1))


class LayerBiasLogger:
    def __init__(self, path):
        self.path = Path(path) if path else None
        if self.path is not None:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            if not self.path.exists() or self.path.stat().st_size == 0:
                with self.path.open("w", newline="") as f:
                    csv.DictWriter(f, fieldnames=LOG_COLUMNS).writeheader()

    def log(self, **row):
        if self.path is None:
            return
        out = {key: row.get(key, "") for key in LOG_COLUMNS}
        needs_header = not self.path.exists() or self.path.stat().st_size == 0
        with self.path.open("a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
            if needs_header:
                writer.writeheader()
            writer.writerow(out)


_LAYER_LOGGERS = {}


def _logger_for(path):
    if not path:
        return LayerBiasLogger(None)
    path = str(path)
    if path not in _LAYER_LOGGERS:
        _LAYER_LOGGERS[path] = LayerBiasLogger(path)
    return _LAYER_LOGGERS[path]


def _record_heuristic_call(self, depth, h_value, current_time):
    depth = int(depth) if depth is not None else -1
    current_time = float(current_time) if current_time is not None else float("nan")
    h_value = float(h_value)

    if getattr(self, "bias_correction", "none") == "online":
        self._layer_online_stats.update(depth, h_value)

    trace = getattr(self, "_layer_current_trace", None)
    if trace is not None:
        trace.append({"depth": depth, "h_value": h_value, "current_time": current_time})

    _logger_for(getattr(self, "layer_log_path", None)).log(
        event_type="heuristic_call",
        problem_id=getattr(self, "problem_id", ""),
        trial_id=getattr(self, "trial_id", ""),
        mode=getattr(self, "bias_correction", "none"),
        iteration=getattr(self, "_layer_iteration", ""),
        depth=depth,
        h_value=h_value,
        current_time=current_time,
        eventual_return="",
    )


def _baseline_for_depth(self, depth):
    mode = getattr(self, "bias_correction", "none")
    depth = int(depth)
    if mode == "offline":
        baseline = getattr(self, "offline_baseline", {}) or {}
        return float(baseline.get(depth, 0.0))
    if mode == "online":
        stats = getattr(self, "_layer_online_stats", None)
        warmup = int(getattr(self, "online_warmup_visits", ONLINE_WARMUP_VISITS))
        if stats is not None and stats.count(depth) >= warmup:
            return float(stats.mean(depth))
    return 0.0


def _uct_display_value(self, anode, parent_snode):
    raw_value = float(anode.value)
    if type(self).__name__ != "C_MCTS":
        return raw_value
    mode = getattr(self, "bias_correction", "none")
    if mode not in {"offline", "online"}:
        return raw_value
    # The action value is a backup from successor nodes, so use the successor layer.
    successor_depth = int(getattr(parent_snode, "depth", 0)) + 1
    return raw_value - _baseline_for_depth(self, successor_depth)


if not getattr(mcts_mod, "_layer_bias_runtime_patched", False):
    mcts_mod._layer_orig_base_search = mcts_mod.Base_MCTS.search
    mcts_mod._layer_orig_base_uct = mcts_mod.Base_MCTS.uct
    mcts_mod._layer_orig_c_init = mcts_mod.C_MCTS.__init__
    mcts_mod._layer_orig_c_uct = mcts_mod.C_MCTS.uct
    mcts_mod._layer_orig_c_heuristic = mcts_mod.C_MCTS.heuristic
    mcts_mod._layer_orig_c_heuristic_init = mcts_mod.C_MCTS.heuristic_init
    mcts_mod._layer_orig_c_create_snode_max = mcts_mod.C_MCTS.create_Snode_max
    mcts_mod._layer_orig_plan = mcts_mod.plan


    def patched_c_init(
        self,
        mdp,
        root_node,
        root_state,
        search_depth,
        exploration_constant,
        stn,
        selection_type,
        k,
        previous_chosen_action_node=None,
        heuristic_name="trpg",
        temporal_heuristic_depth=25,
        temporal_heuristic_strategy="baseline",
        root_baseline_cache=None,
        value_mode="tp_mcts",
        uct_initial_k=3,
        bias_correction="none",
        offline_baseline=None,
        layer_log_path=None,
        online_stats=None,
        online_warmup_visits=100,
        problem_id="",
        trial_id="",
    ):
        if bias_correction not in {"none", "offline", "online"}:
            raise ValueError("bias_correction must be one of: none, offline, online")
        # Set logging/correction attributes before the original constructor creates
        # max-initialized child nodes, because those paths call heuristic_init().
        self.bias_correction = bias_correction
        self.offline_baseline = {int(k): float(v) for k, v in (offline_baseline or {}).items()}
        self.layer_log_path = str(layer_log_path) if layer_log_path else None
        self._layer_online_stats = online_stats if online_stats is not None else RunningDepthStats()
        self.online_warmup_visits = int(online_warmup_visits)
        self.problem_id = problem_id
        self.trial_id = trial_id
        self._layer_current_trace = None
        self._layer_iteration = ""
        self._layer_create_depth = None
        mcts_mod._layer_orig_c_init(
            self,
            mdp,
            root_node,
            root_state,
            search_depth,
            exploration_constant,
            stn,
            selection_type,
            k,
            previous_chosen_action_node=previous_chosen_action_node,
            heuristic_name=heuristic_name,
            temporal_heuristic_depth=temporal_heuristic_depth,
            temporal_heuristic_strategy=temporal_heuristic_strategy,
            root_baseline_cache=root_baseline_cache,
            value_mode=value_mode,
            uct_initial_k=uct_initial_k,
        )


    def patched_base_uct(self, snode, explore_constant):
        anodes = snode.children
        best_ub = -float("inf")
        best_action = -1
        for action in snode.possible_actions:
            if anodes[action].count == 0:
                return action
            ub = _uct_display_value(self, anodes[action], snode) + (
                explore_constant * math.sqrt(math.log(snode.count) / anodes[action].count)
            )
            if ub > best_ub:
                best_ub = ub
                best_action = action
        assert best_action != -1
        return best_action


    def patched_c_uct(self, snode, explore_constant):
        if self._uct_filter_mode is None:
            return mcts_mod.Base_MCTS.uct(self, snode, explore_constant)
        anodes = snode.children
        candidate_actions = self._allowed_actions_for_uct(snode)
        if not candidate_actions:
            return mcts_mod.Base_MCTS.uct(self, snode, explore_constant)
        best_ub = -float("inf")
        best_action = None
        for action in candidate_actions:
            if anodes[action].count == 0:
                return action
            ub = _uct_display_value(self, anodes[action], snode) + (
                explore_constant * math.sqrt(math.log(snode.count) / anodes[action].count)
            )
            if ub > best_ub:
                best_ub = ub
                best_action = action
        if best_action is not None:
            return best_action
        return mcts_mod.Base_MCTS.uct(self, snode, explore_constant)


    def patched_search(self, timeout=1, selection_type="avg"):
        start_time = time.time()
        current_time = time.time()
        i = 0
        avg_variants = {"avg", "avg_topk", "avg_pw"}
        selection = (
            self.selection
            if selection_type in avg_variants
            else (self.selection_root_interval if selection_type == "rootInterval" else self.selection_max)
        )
        while current_time < start_time + timeout:
            is_c_mcts = type(self).__name__ == "C_MCTS"
            if is_c_mcts:
                self._layer_iteration = i
                self._layer_current_trace = []
            ret = selection(self.root_node)
            if is_c_mcts:
                trace = self._layer_current_trace or []
                for entry in trace:
                    _logger_for(getattr(self, "layer_log_path", None)).log(
                        event_type="simulation_end",
                        problem_id=getattr(self, "problem_id", ""),
                        trial_id=getattr(self, "trial_id", ""),
                        mode=getattr(self, "bias_correction", "none"),
                        iteration=i,
                        depth=entry["depth"],
                        h_value=entry["h_value"],
                        current_time=entry["current_time"],
                        eventual_return=float(ret) if isinstance(ret, (int, float)) and math.isfinite(float(ret)) else ret,
                    )
                self._layer_current_trace = None
            current_time = time.time()
            i += 1
        self._last_search_iterations = i
        return self.best_action(self.root_node)


    def patched_create_snode_max(self, state, depth, stn, parent=None, previous_chosen_action_node=None):
        previous_depth = getattr(self, "_layer_create_depth", None)
        self._layer_create_depth = depth
        try:
            return mcts_mod._layer_orig_c_create_snode_max(
                self,
                state,
                depth,
                stn,
                parent,
                previous_chosen_action_node,
            )
        finally:
            self._layer_create_depth = previous_depth


    def patched_heuristic(self, snode):
        current_time = 0
        if snode.parent:
            current_time = snode.parent.stn.get_current_end_time()
        score = mcts_mod._layer_orig_c_heuristic(self, snode)
        _record_heuristic_call(self, getattr(snode, "depth", -1), score, current_time)
        return score


    def patched_heuristic_init(self, state, stn):
        score = mcts_mod._layer_orig_c_heuristic_init(self, state, stn)
        base_depth = getattr(self, "_layer_create_depth", None)
        depth = (int(base_depth) + 1) if base_depth is not None else -1
        current_time = stn.get_current_end_time()
        _record_heuristic_call(self, depth, score, current_time)
        return score


    def _tree_metrics(root_node):
        if root_node is None:
            return {"max_tree_depth": 0, "root_branching_visited": 0, "root_branching_total": 0}
        max_depth = int(getattr(root_node, "depth", 0))
        stack = [root_node]
        seen = set()
        while stack:
            node = stack.pop()
            node_id = id(node)
            if node_id in seen:
                continue
            seen.add(node_id)
            max_depth = max(max_depth, int(getattr(node, "depth", max_depth)))
            for anode in getattr(node, "children", {}).values():
                for child in getattr(anode, "children", {}).values():
                    stack.append(child)
        root_children = getattr(root_node, "children", {})
        visited = sum(1 for child in root_children.values() if getattr(child, "count", 0) > 0)
        return {
            "max_tree_depth": max_depth,
            "root_branching_visited": visited,
            "root_branching_total": len(root_children),
        }


    def patched_plan(
        mdp,
        steps,
        search_time,
        search_depth,
        exploration_constant,
        selection_type="avg",
        k=10,
        heuristic_name="trpg",
        temporal_heuristic_depth=25,
        temporal_heuristic_strategy="baseline",
        value_mode="tp_mcts",
        bias_correction="none",
        offline_baseline=None,
        layer_log_path=None,
        online_warmup_visits=100,
        problem_id="",
        trial_id="",
    ):
        stn = create_init_stn(mdp)
        root_state = mdp.initial_state()
        reuse = False
        history = []
        previous_action_node = None
        step = 0
        root_node = None
        online_stats = RunningDepthStats()
        use_baseline_cache = (
            heuristic_name == "temporal_probabilistic_rpg"
            and temporal_heuristic_strategy == "baseline_cached"
        )
        baseline_cache_table = None

        while stn.get_current_end_time() <= mdp.deadline() and step < steps:
            mcts = mcts_mod.C_MCTS(
                mdp,
                root_node,
                root_state,
                search_depth,
                exploration_constant,
                stn,
                selection_type,
                k,
                previous_action_node,
                heuristic_name=heuristic_name,
                temporal_heuristic_depth=temporal_heuristic_depth,
                temporal_heuristic_strategy=temporal_heuristic_strategy,
                root_baseline_cache=baseline_cache_table,
                value_mode=value_mode,
                bias_correction=bias_correction,
                offline_baseline=offline_baseline,
                layer_log_path=layer_log_path,
                online_stats=online_stats,
                online_warmup_visits=online_warmup_visits,
                problem_id=problem_id,
                trial_id=trial_id,
            )
            action = mcts.search(search_time, selection_type)
            metrics = _tree_metrics(mcts.root_node)
            metrics.update({
                "problem_id": problem_id,
                "trial_id": trial_id,
                "mode": bias_correction,
                "planning_step": step,
                "search_iterations": getattr(mcts, "_last_search_iterations", 0),
            })
            LAYER_BIAS_EPISODE_METRICS.append(metrics)

            if action == -1:
                return 0, -math.inf

            terminal, root_state, reward = mcts.mdp.step(root_state, action)

            if reuse and root_state in mcts.root_node.children[action].children:
                root_node = mcts.root_node.children[action].children[root_state]
                root_node.set_depth(0)

            action_node = mcts.root_node.children[action] if selection_type == "rootInterval" else None
            previous_action_node = update_stn(
                stn,
                action,
                previous_action_node,
                type="SetTime",
                action_node=action_node,
            )
            assert stn.is_consistent()
            history.append(previous_action_node)

            if terminal:
                return 1, stn.get_current_end_time()

            if use_baseline_cache:
                current_time = stn.get_current_end_time()
                _, baseline_cache_table = mcts_mod._tprpg_heuristic_value(
                    mdp,
                    root_state,
                    current_time,
                    temporal_heuristic_depth,
                    temporal_heuristic_strategy,
                    cached_table=baseline_cache_table,
                    leaf_heuristic_name=heuristic_name,
                )

            step += 1

        return 0, -math.inf


    mcts_mod.C_MCTS.__init__ = patched_c_init
    mcts_mod.Base_MCTS.uct = patched_base_uct
    mcts_mod.C_MCTS.uct = patched_c_uct
    mcts_mod.Base_MCTS.search = patched_search
    mcts_mod.C_MCTS.create_Snode_max = patched_create_snode_max
    mcts_mod.C_MCTS.heuristic = patched_heuristic
    mcts_mod.C_MCTS.heuristic_init = patched_heuristic_init
    mcts_mod.plan = patched_plan
    mcts_mod._layer_bias_runtime_patched = True

print("Layer-bias runtime patches installed.")

In [ ]:
# Cell 3 - Benchmark helpers and Phase 1 instrumentation run
import contextlib
import io
import json
import statistics
from types import SimpleNamespace

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import unified_planning as up
from unified_planning.shortcuts import *  # noqa: F401,F403 - required by local domain constructors.
import unified_planning.domains
from unified_planning.engines.convert_problem import Convert_problem
from unified_planning.engines.mdp import MDP
import unified_planning.engines.solvers.greedy_parallel as greedy_parallel

# Some heuristic helpers read optional CLI attributes from up.args. Keep defaults explicit in notebooks.
up.args = SimpleNamespace(
    resolution_alpha=2.0,
    resolution_forced_minimum=False,
    resolution_reference_t=None,
)

DOMAIN_CLASSES = {
    "machine_shop": up.domains.Machine_Shop,
    "nasa_rover": up.domains.Nasa_Rover,
    "stuck_car_1o": up.domains.Stuck_Car_1o,
    "stuck_car": up.domains.Stuck_Car,
    "conc": up.domains.Conc,
    "full_conc": up.domains.Full_Conc,
    "prob_conc": up.domains.Prob_Conc,
    "best_no_parallel": up.domains.Best_No_Parallel,
    "simple": up.domains.Simple,
    "hosting": up.domains.Hosting,
    "prob_match_cellar": up.domains.Prob_MatchCellar,
}

PROBLEM_SPECS = [
    {"problem_id": "nasa_o2_d25", "domain": "nasa_rover", "domain_type": "regular", "deadline": 25, "object_amount": 2, "garbage_amount": 0},
    {"problem_id": "nasa_o2_d35", "domain": "nasa_rover", "domain_type": "regular", "deadline": 35, "object_amount": 2, "garbage_amount": 0},
    {"problem_id": "nasa_o2_d45", "domain": "nasa_rover", "domain_type": "regular", "deadline": 45, "object_amount": 2, "garbage_amount": 0},
    {"problem_id": "nasa_o3_d25", "domain": "nasa_rover", "domain_type": "regular", "deadline": 25, "object_amount": 3, "garbage_amount": 0},
    {"problem_id": "nasa_o3_d35", "domain": "nasa_rover", "domain_type": "regular", "deadline": 35, "object_amount": 3, "garbage_amount": 0},
    {"problem_id": "nasa_o3_d45", "domain": "nasa_rover", "domain_type": "regular", "deadline": 45, "object_amount": 3, "garbage_amount": 0},
]
ACTIVE_PROBLEM_SPECS = PROBLEM_SPECS[:N_PROBLEMS] if N_PROBLEMS else PROBLEM_SPECS


def set_trial_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


def build_regular_mdp(spec):
    domain_name = spec["domain"]
    model_cls = DOMAIN_CLASSES[domain_name]
    model = model_cls(
        kind=spec.get("domain_type", "regular"),
        deadline=spec["deadline"],
        object_amount=spec.get("object_amount", 2),
        garbage_amount=spec.get("garbage_amount", 0),
    )
    if domain_name == "nasa_rover":
        grounder = up.engines.compilers.Grounder(model.grounding_map())
    else:
        grounder = up.engines.compilers.Grounder()
    ground_problem = grounder._compile(model.problem).problem
    converted_problem = Convert_problem(ground_problem)._converted_problem
    return MDP(
        converted_problem,
        discount_factor=DISCOUNT_FACTOR,
        reward_mode=REWARD_MODE,
        step_penalty=STEP_PENALTY,
    )


def _summarize_tree_metrics(problem_id, mode, trial_id):
    rows = [
        row for row in LAYER_BIAS_EPISODE_METRICS
        if row.get("problem_id") == problem_id and row.get("mode") == mode and row.get("trial_id") == trial_id
    ]
    if not rows:
        return {"avg_depth_reached": 0.0, "avg_root_branching": 0.0, "avg_search_iterations": 0.0}
    return {
        "avg_depth_reached": float(np.mean([r["max_tree_depth"] for r in rows])),
        "avg_root_branching": float(np.mean([r["root_branching_visited"] for r in rows])),
        "avg_search_iterations": float(np.mean([r["search_iterations"] for r in rows])),
    }


def run_one_planner_trial(mdp, spec, trial_idx, mode, planner="mcts", offline_baseline=None, log_path=None):
    trial_seed = SEED + trial_idx
    set_trial_seed(trial_seed)
    problem_id = spec["problem_id"]
    with contextlib.redirect_stdout(io.StringIO()):
        if planner == "greedy":
            success, finish_time = greedy_parallel.plan(
                mdp,
                STEP_LIMIT,
                SEARCH_TIME,
                SEARCH_DEPTH,
                EXPLORATION_CONSTANT,
                SELECTION_TYPE,
                K,
                HEURISTIC_NAME,
                TEMPORAL_HEURISTIC_DEPTH,
                TEMPORAL_STRATEGY,
            )
        else:
            success, finish_time = mcts_mod.plan(
                mdp,
                STEP_LIMIT,
                SEARCH_TIME,
                SEARCH_DEPTH,
                EXPLORATION_CONSTANT,
                SELECTION_TYPE,
                K,
                HEURISTIC_NAME,
                TEMPORAL_HEURISTIC_DEPTH,
                TEMPORAL_STRATEGY,
                value_mode="tp_mcts",
                bias_correction=mode,
                offline_baseline=offline_baseline,
                layer_log_path=log_path,
                online_warmup_visits=ONLINE_WARMUP_VISITS,
                problem_id=problem_id,
                trial_id=trial_idx,
            )
    metrics = _summarize_tree_metrics(problem_id, mode, trial_idx) if planner != "greedy" else {
        "avg_depth_reached": float("nan"),
        "avg_root_branching": float("nan"),
        "avg_search_iterations": float("nan"),
    }
    return {
        "problem_id": problem_id,
        "mode": mode if planner != "greedy" else "greedy",
        "trial_id": trial_idx,
        "success": int(success),
        "finish_time": finish_time,
        **metrics,
    }


def run_benchmark_suite(mode="none", offline_baseline=None, log_path=None, include_greedy=False, run_label=""):
    LAYER_BIAS_EPISODE_METRICS.clear()
    rows = []
    specs = ACTIVE_PROBLEM_SPECS
    outer = tqdm(specs, desc=f"{run_label or mode}: problems")
    for spec in outer:
        mdp = build_regular_mdp(spec)
        for trial_idx in tqdm(range(N_RUNS), leave=False, desc=spec["problem_id"]):
            rows.append(run_one_planner_trial(mdp, spec, trial_idx, mode, "mcts", offline_baseline, log_path))
        if include_greedy:
            for trial_idx in tqdm(range(N_RUNS), leave=False, desc=spec["problem_id"] + " greedy"):
                rows.append(run_one_planner_trial(mdp, spec, trial_idx, "greedy", "greedy", None, None))
    return pd.DataFrame(rows)

phase1_log_path = Path("/content/logs_phase1.csv") if Path("/content").exists() else ARTIFACTS_DIR / "logs_phase1.csv"
if phase1_log_path.exists():
    phase1_log_path.unlink()

phase1_scores = run_benchmark_suite(mode="none", log_path=phase1_log_path, include_greedy=False, run_label="Phase 1")
phase1_scores_path = ARTIFACTS_DIR / "phase1_scores.csv"
phase1_scores.to_csv(phase1_scores_path, index=False)

# Keep the requested /content path and also copy into artifacts for zipping.
phase1_artifact_log = ARTIFACTS_DIR / "logs_phase1.csv"
pd.read_csv(phase1_log_path).to_csv(phase1_artifact_log, index=False)

print(f"Phase 1 score rows: {len(phase1_scores)}")
print(f"Phase 1 log path: {phase1_log_path}")
print(phase1_scores.groupby("problem_id")["success"].mean().rename("score"))

In [ ]:
# Cell 4 - Phase 2 diagnosis plots
import matplotlib.pyplot as plt

phase1_log = pd.read_csv(phase1_log_path)
heuristic_calls = phase1_log[phase1_log["event_type"] == "heuristic_call"].copy()
simulation_rows = phase1_log[phase1_log["event_type"] == "simulation_end"].copy()

for df in (heuristic_calls, simulation_rows):
    df["depth"] = pd.to_numeric(df["depth"], errors="coerce")
    df["h_value"] = pd.to_numeric(df["h_value"], errors="coerce")
    df["eventual_return"] = pd.to_numeric(df["eventual_return"], errors="coerce")

bias_curve = (
    heuristic_calls.dropna(subset=["depth", "h_value"])
    .groupby("depth", as_index=True)["h_value"]
    .agg(["mean", "std", "count"])
    .sort_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(bias_curve.index, bias_curve["mean"], marker="o")
axes[0].set_title("Mean h by tree depth: B(d)")
axes[0].set_xlabel("Depth")
axes[0].set_ylabel("Mean heuristic value")
axes[0].grid(True, alpha=0.3)

axes[1].plot(bias_curve.index, bias_curve["std"].fillna(0.0), marker="o", color="tab:orange")
axes[1].set_title("Std h by tree depth")
axes[1].set_xlabel("Depth")
axes[1].set_ylabel("Std heuristic value")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "phase2_bias_curve_and_std.png", dpi=160)
plt.show()

baseline_by_depth = bias_curve["mean"].to_dict()
simulation_rows["h_minus_B"] = simulation_rows.apply(
    lambda row: row["h_value"] - baseline_by_depth.get(row["depth"], 0.0),
    axis=1,
)

corr_rows = []
for depth, group in simulation_rows.dropna(subset=["eventual_return", "h_value"]).groupby("depth"):
    if len(group) < 3 or group["eventual_return"].nunique() < 2:
        corr_h = np.nan
        corr_centered = np.nan
    else:
        corr_h = group["h_value"].corr(group["eventual_return"])
        corr_centered = group["h_minus_B"].corr(group["eventual_return"])
    corr_rows.append({"depth": depth, "corr_h": corr_h, "corr_h_minus_B": corr_centered, "n": len(group)})

corr_df = pd.DataFrame(corr_rows).sort_values("depth")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(corr_df["depth"], corr_df["corr_h"], marker="o", label="corr(h, return)")
ax.plot(corr_df["depth"], corr_df["corr_h_minus_B"], marker="o", label="corr(h - B(d), return)")
ax.axhline(0.0, color="black", linewidth=1, alpha=0.4)
ax.set_title("Per-depth calibration correlation")
ax.set_xlabel("Depth")
ax.set_ylabel("Pearson correlation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "phase2_depth_correlation.png", dpi=160)
plt.show()

bias_curve.to_csv(ARTIFACTS_DIR / "phase2_bias_curve.csv")
corr_df.to_csv(ARTIFACTS_DIR / "phase2_correlations.csv", index=False)

print("Bias curve head:")
display(bias_curve.head(12))
print("Correlation head:")
display(corr_df.head(12))

In [ ]:
# Cell 5 - Phase 3 derive offline baseline B(d)
window = 3
offline_baseline_series = (
    bias_curve["mean"]
    .sort_index()
    .rolling(window=window, center=True, min_periods=1)
    .mean()
)
offline_baseline = {int(depth): float(value) for depth, value in offline_baseline_series.items()}

baseline_path = ARTIFACTS_DIR / "offline_baseline_Bd.json"
with baseline_path.open("w") as f:
    json.dump(offline_baseline, f, indent=2, sort_keys=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bias_curve.index, bias_curve["mean"], marker="o", label="Raw mean h")
ax.plot(offline_baseline_series.index, offline_baseline_series.values, marker="o", label="Smoothed B(d), window=3")
ax.set_title("Offline layer baseline")
ax.set_xlabel("Depth")
ax.set_ylabel("Heuristic baseline")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "phase3_offline_baseline.png", dpi=160)
plt.show()

print(f"Saved offline baseline to {baseline_path}")
offline_baseline

In [ ]:
# Cell 6 - Phase 4 three-way comparison plus greedy baseline
comparison_frames = []
comparison_log_paths = {}

for mode in ["none", "offline", "online"]:
    log_path = ARTIFACTS_DIR / f"logs_phase4_{mode}.csv"
    if log_path.exists():
        log_path.unlink()
    baseline_arg = offline_baseline if mode == "offline" else None
    df_mode = run_benchmark_suite(
        mode=mode,
        offline_baseline=baseline_arg,
        log_path=log_path,
        include_greedy=False,
        run_label=f"Phase 4 {mode}",
    )
    df_mode.to_csv(ARTIFACTS_DIR / f"phase4_scores_{mode}.csv", index=False)
    comparison_frames.append(df_mode)
    comparison_log_paths[mode] = log_path

# Greedy baseline: same MDP builder, heuristic, seeds, and problem list; no MCTS tree metrics.
greedy_rows = []
for spec in tqdm(ACTIVE_PROBLEM_SPECS, desc="Greedy baseline: problems"):
    mdp = build_regular_mdp(spec)
    for trial_idx in tqdm(range(N_RUNS), leave=False, desc=spec["problem_id"]):
        greedy_rows.append(run_one_planner_trial(mdp, spec, trial_idx, "greedy", planner="greedy"))

greedy_df = pd.DataFrame(greedy_rows)
greedy_df.to_csv(ARTIFACTS_DIR / "phase4_scores_greedy.csv", index=False)
comparison_frames.append(greedy_df)

comparison_results = pd.concat(comparison_frames, ignore_index=True)
comparison_results.to_csv(ARTIFACTS_DIR / "phase4_all_scores.csv", index=False)

print("Run-level rows:", len(comparison_results))
display(comparison_results.head())

In [ ]:
# Cell 7 - Results table and plots
score_table = (
    comparison_results
    .groupby(["problem_id", "mode"], as_index=False)["success"]
    .mean()
    .pivot(index="problem_id", columns="mode", values="success")
    .rename(columns={
        "none": "score_none",
        "offline": "score_offline",
        "online": "score_online",
        "greedy": "score_greedy",
    })
)

ordered_cols = ["score_none", "score_offline", "score_online", "score_greedy"]
score_table = score_table[[col for col in ordered_cols if col in score_table.columns]]
score_table.to_csv(ARTIFACTS_DIR / "phase4_problem_score_table.csv")

print("Per-problem scores")
display(score_table)

mode_scores = comparison_results.groupby(["mode", "problem_id"], as_index=False)["success"].mean()
mean_by_mode = mode_scores.groupby("mode")["success"].mean().reindex(["none", "offline", "online", "greedy"])
stderr_by_mode = mode_scores.groupby("mode")["success"].sem().reindex(mean_by_mode.index).fillna(0.0)

print("Mean score per mode")
display(mean_by_mode.rename("mean_score"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(mean_by_mode.index, mean_by_mode.values, yerr=stderr_by_mode.values, capsize=4)
ax.set_title("Score comparison by mode")
ax.set_xlabel("Mode")
ax.set_ylabel("Mean success score")
ax.set_ylim(0, 1.05)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "phase4_mode_score_bar.png", dpi=160)
plt.show()

mcts_metrics = comparison_results[comparison_results["mode"].isin(["none", "offline", "online"])]
metric_summary = (
    mcts_metrics
    .groupby("mode")[["avg_depth_reached", "avg_root_branching", "avg_search_iterations"]]
    .mean()
    .reindex(["none", "offline", "online"])
)
metric_summary.to_csv(ARTIFACTS_DIR / "phase4_tree_shape_metrics.csv")
print("Tree-shape metrics")
display(metric_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
metric_summary["avg_depth_reached"].plot(kind="bar", ax=axes[0], title="Avg depth reached")
axes[0].set_ylabel("Depth")
metric_summary["avg_root_branching"].plot(kind="bar", ax=axes[1], title="Avg visited root branching")
axes[1].set_ylabel("Visited root actions")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "phase4_tree_shape_bar.png", dpi=160)
plt.show()

# Cell 8 - Verdict

After running the notebook top-to-bottom, fill in the verdict from the generated tables and plots:

1. **Is the layer-bias curve real and monotone?** Inspect `phase2_bias_curve_and_std.png`. A monotone drift in `Mean h by tree depth` supports the layer-bias diagnosis. A large or depth-dependent standard deviation suggests that plain subtraction may be too coarse and that a z-score variant should be tested separately.
2. **Does subtracting B(d) close the greedy gap?** Compare `score_none`, `score_offline`, `score_online`, and `score_greedy` in the per-problem table. The greedy gap is closed only if the corrected MCTS modes approach the greedy baseline across most problems, not just one easy instance.
3. **Does online beat offline?** Compare `score_online` vs `score_offline` and the tree-shape metrics. Online should be preferred only if it improves scores without making the tree shallower or less exploratory.

Important caveat: if UCT subtracts the same scalar baseline for every sibling at a fixed parent depth, the UCT argmax is mathematically unchanged. If the corrected modes match `none`, that does not disprove the bias curve; it suggests the correction must be state- or successor-dependent, or applied as an incremental potential difference, while still keeping raw heuristic semantics for backups.

In [ ]:
# Final cell - Zip artifacts for download
import shutil

zip_base = Path("/content/layer_bias_artifacts") if Path("/content").exists() else (ARTIFACTS_DIR.parent / "layer_bias_artifacts")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(ARTIFACTS_DIR))
print(f"Created artifact archive: {zip_path}")

if IN_COLAB:
    from google.colab import files  # type: ignore
    files.download(zip_path)